## Tactical Development


### Dataset :: (fake dataset - databricks market place)



- "databricks_simulated_retail_customer_data"

1) The first step is to explore the dataset as much as possible, to identify all the possible relationships existent between the tables and volumes;

2) Ideally more than one architecture for data modeling, until "best" pipeline and insight is found;


## V01

### New tables

##### creating table from VOLUME

In [0]:
# %sql
# Sample Script 
# CREATE SCHEMA IF NOT EXISTS medallion_catalog.db_bronze
# MANAGED LOCATION 'abfss://catalog@container.dfs.core.windows.net/folder';

# USE CATALOG medallion_catalog;
# USE SCHEMA db_bronze;

# DROP TABLE IF EXISTS medallion_catalog.db_bronze.companies;

# CREATE TABLE IF NOT EXISTS medallion_catalog.db_bronze.companies 
# AS
# SELECT company_name, founded_date, country
#   FROM read_files('/Volumes/medallion_catalog/db_landing/vol_medallion/top_tech_companies/',
#   format => 'csv',
#   header => true);


#### creating tables from tables

In [0]:
# %sql
# -- 1. Create the table structure with an IDENTITY column
# CREATE TABLE IF NOT EXISTS main.default.user_profile (
#   user_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
#   user_name STRING,
#   signup_date DATE,
#   account_balance DECIMAL(18, 2),
#   is_active BOOLEAN
# )
# USING DELTA;

# -- 2. Populate the table (excluding the identity column)
# INSERT INTO main.default.user_profile (user_name, signup_date, account_balance, is_active)
# SELECT 
#   name, 
#   CAST(registration_time AS DATE), 
#   CAST(balance AS DECIMAL(18, 2)), 
#   true
# FROM main.staging.raw_users;

## Tables

### Customers ::

##### Table Customers address ::

In [0]:
%sql
SELECT
customer_id,
state,
city,
postcode,
street,
number,
unit,
region,
district
from databricks_simulated_retail_customer_data.v01.customers limit 5


In [0]:
%sql
-- 1. Define the table structure with types and Identity column
CREATE TABLE IF NOT EXISTS end_to_end_retail.db_landing.customer_address (
  -- customer_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  customer_id BIGINT,
  state STRING,
  city STRING,
  postcode STRING,
  street STRING,
  number STRING,
  unit STRING,
  region STRING,
  district STRING
)
USING DELTA;

-- 2. Insert data from your source (omit customer_id to let it auto-generate)
INSERT INTO end_to_end_retail.db_landing.customer_address (
  state, 
  city, 
  postcode, 
  street, 
  number, 
  unit, 
  region, 
  district
)
SELECT 
  state, 
  city, 
  postcode, 
  street, 
  number, 
  unit, 
  region, 
  district
FROM databricks_simulated_retail_customer_data.v01.customers;

##### Table Customers geo_loc ::

In [0]:
 %sql
SELECT 
 customer_id,
 lon,
 lat,
 ship_to_address,
 valid_from,
 valid_to
FROM databricks_simulated_retail_customer_data.v01.customers;

In [0]:
 %sql
 CREATE TABLE IF NOT EXISTS end_to_end_retail.db_landing.customer_geoloc (
 customer_id BIGINT,
 lon STRING,
 lat STRING,
 ship_to_address STRING,
 valid_from STRING,
 valid_to STRING
)
USING DELTA;

 -- 2. Insert data from your source (omit customer_id to let it auto-generate)
INSERT INTO end_to_end_retail.db_landing.customer_geoloc (
 customer_id,
 lon,
 lat,
 ship_to_address,
 valid_from,
 valid_to
)
SELECT 
 customer_id,
 lon,
 lat,
 ship_to_address,
 valid_from,
 valid_to
FROM databricks_simulated_retail_customer_data.v01.customers;

##### Table Customers tax_id ::

In [0]:
%sql
select 
customer_id,
customer_name,
tax_id,
tax_code,
units_purchased,
loyalty_segment
from databricks_simulated_retail_customer_data.v01.customers limit 5

In [0]:
%sql
CREATE TABLE IF NOT EXISTS end_to_end_retail.db_landing.customer_taxid (
customer_id BIGINT,
customer_name STRING,
tax_id STRING,
tax_code STRING,
units_purchased STRING,
loyalty_segment STRING
)
USING DELTA;

 -- 2. Insert data from your source (omit customer_id to let it auto-generate)
INSERT INTO end_to_end_retail.db_landing.customer_taxid (
customer_id,
customer_name,
tax_id,
tax_code,
units_purchased,
loyalty_segment
)
select 
customer_id,
customer_name,
tax_id,
tax_code,
units_purchased,
loyalty_segment
from databricks_simulated_retail_customer_data.v01.customers;

### Sales ::

In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    order_date,
    product_category,
    total_price,
    -- ai_query("databricks-meta-llama-3-3-70b-instruct", 'Apply NLP to extract only meaningful description 5 word max: ' || product_name) as response
    -- ai_query("databricks-meta-llama-3-3-70b-instruct", 'Summarize the product name only 5 words:' || product_name) as response,
    product_name,
    get_json_object(product, '$.curr') AS curr,
    get_json_object(product, '$.id') AS id,
    get_json_object(product, '$.name') AS product_name,
    get_json_object(product, '$.price') AS price,
    get_json_object(product, '$.qty') AS qty,
    get_json_object(product, '$.unit') AS unit
FROM (
  SELECT * FROM databricks_simulated_retail_customer_data.v01.sales
) t
limit 5

In [0]:
%sql
CREATE TABLE IF NOT EXISTS end_to_end_retail.db_landing.sales (
customer_id LONG,
customer_name STRING,
order_date DATE,
product_category STRING,
total_price LONG,
product_name STRING,
curr STRING,
id STRING,
product STRING,
price STRING,
qty STRING,
unit STRING
)
USING DELTA;

 -- 2. Insert data from your source (omit customer_id to let it auto-generate)
INSERT INTO end_to_end_retail.db_landing.sales (
customer_id,
customer_name,
order_date,
product_category,
total_price,
product_name,
curr,
id,
product,
price,
qty,
unit
)
SELECT
    customer_id,
    customer_name,
    order_date,
    product_category,
    total_price,
    -- ai_query("databricks-meta-llama-3-3-70b-instruct", 'Apply NLP to extract only meaningful description 5 word max: ' || product_name) as response
    -- ai_query("databricks-meta-llama-3-3-70b-instruct", 'Summarize the product name only 5 words:' || product_name) as response,
    product_name,
    get_json_object(product, '$.curr') AS curr,
    get_json_object(product, '$.id') AS id,
    get_json_object(product, '$.name') AS product,
    get_json_object(product, '$.price') AS price,
    get_json_object(product, '$.qty') AS qty,
    get_json_object(product, '$.unit') AS unit
FROM (
  SELECT * FROM databricks_simulated_retail_customer_data.v01.sales
) t



### Sales_order ::

#### Exploded the parse_item_value

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StructType, StructField, StringType
import json

schema = ArrayType(StructType([
    StructField("item", StringType(), True),
    StructField("value", StringType(), True)
]))

def parse_item_value_udf(json_array):
    arr = json.loads(json_array)
    result = []
    for pair in arr:
        if isinstance(pair, list) and len(pair) == 2:
            result.append({"item": str(pair[0]), "value": str(pair[1])})
    return result

spark.udf.register("end_to_end_retail.db_landing.parse_item_value", parse_item_value_udf, schema)

In [0]:
%sql
SELECT
  exploded.item,
  exploded.value
FROM (
  SELECT explode(parse_item_value(clicked_items)) AS exploded
  FROM databricks_simulated_retail_customer_data.v01.sales_orders
  LIMIT 3
) t

In [0]:
%sql
select 
customer_id,
customer_name,
number_of_line_items,
order_datetime,
order_number,
ordered_products,
promo_info,
-- clicked_items,
exploded.item,
exploded.value
FROM (
  SELECT
    customer_id,
    customer_name,
    number_of_line_items,
    order_datetime,
    order_number,
    ordered_products,
    promo_info,
    clicked_items,
    explode(parse_item_value(clicked_items)) AS exploded
  FROM databricks_simulated_retail_customer_data.v01.sales_orders
) t
WHERE order_number = '317568014'

#### Exploded the promo info

In [0]:
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, DoubleType

promo_schema = ArrayType(StructType([
    StructField("promo_disc", DoubleType(), True),
    StructField("promo_id", StringType(), True),
    StructField("promo_item", StringType(), True),
    StructField("promo_qty", StringType(), True)
]))

def parse_promo_info_udf(json_array):
    import json
    arr = json.loads(json_array)
    result = []
    for obj in arr:
        result.append({
            "promo_disc": float(obj.get("promo_disc", 0)),
            "promo_id": str(obj.get("promo_id", "")),
            "promo_item": str(obj.get("promo_item", "")),
            "promo_qty": str(obj.get("promo_qty", ""))
        })
    return result

spark.udf.register("end_to_end_retail.db_landing.parse_promo_info", parse_promo_info_udf, promo_schema)

#### Exploded ordered products

In [0]:
from pyspark.sql.types import ArrayType, StructType, StructField, StringType

product_schema = ArrayType(StructType([
    StructField("curr", StringType(), True),
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("price", StringType(), True),
    StructField("promotion_info", StringType(), True),
    StructField("qty", StringType(), True),
    StructField("unit", StringType(), True)
]))

def parse_product_list_udf(json_array):
    import json
    arr = json.loads(json_array)
    result = []
    for obj in arr:
        result.append({
            "curr": obj.get("curr", ""),
            "id": obj.get("id", ""),
            "name": obj.get("name", ""),
            "price": obj.get("price", ""),
            "promotion_info": obj.get("promotion_info", ""),
            "qty": obj.get("qty", ""),
            "unit": obj.get("unit", "")
        })
    return result

spark.udf.register("parse_product_list", parse_product_list_udf, product_schema)


#### Table Clicked Items

In [0]:
%sql
select 
customer_id,
exploded.item,
exploded.value

FROM (
  SELECT
    customer_id,
    customer_name,
    number_of_line_items,
    order_datetime,
    order_number,
    ordered_products,
    promo_info,
    clicked_items,
    explode(parse_item_value(clicked_items)) AS exploded
  FROM databricks_simulated_retail_customer_data.v01.sales_orders
) t
where customer_id = '14939501'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS end_to_end_retail.db_landing.clicked_items (
customer_id BIGINT,
item STRING,
value STRING
)
USING DELTA;

 -- 2. Insert data from your source (omit customer_id to let it auto-generate)
INSERT INTO end_to_end_retail.db_landing.clicked_items (
select 
customer_id,
exploded.item,
exploded.value

FROM (
  SELECT
    customer_id,
    customer_name,
    number_of_line_items,
    order_datetime,
    order_number,
    ordered_products,
    promo_info,
    clicked_items,
    explode(parse_item_value(clicked_items)) AS exploded
  FROM databricks_simulated_retail_customer_data.v01.sales_orders
) t
)

#### Table Promo info

In [0]:
%sql
SELECT
  customer_id,
  order_number,
  exploded.promo_disc,
  exploded.promo_id,
  exploded.promo_item,
  exploded.promo_qty
FROM (
  SELECT
    customer_id,
    order_number,
    explode(parse_promo_info(promo_info)) AS exploded
  FROM databricks_simulated_retail_customer_data.v01.sales_orders
  WHERE promo_info IS NOT NULL
  LIMIT 5
) t

In [0]:
%sql
CREATE TABLE IF NOT EXISTS end_to_end_retail.db_landing.promo_info (
  customer_id BIGINT,
  order_number STRING,
  promo_disc STRING,
  promo_id STRING,
  promo_item STRING,
  promo_qty STRING
)
USING DELTA;

-- 2. Insert data from your source
INSERT INTO end_to_end_retail.db_landing.promo_info (
  customer_id,
  order_number,
  promo_disc,
  promo_id,
  promo_item,
  promo_qty
)
SELECT
  customer_id,
  order_number,
  exploded.promo_disc AS promo_disc,
  exploded.promo_id AS promo_id,
  exploded.promo_item AS promo_item,
  exploded.promo_qty AS promo_qty
FROM (
  SELECT
    customer_id,
    order_number,
    explode(parse_promo_info(promo_info)) AS exploded
  FROM databricks_simulated_retail_customer_data.v01.sales_orders  
) t

#### Table Ordered products

In [0]:
%sql
SELECT
  customer_id,
  order_number,
  exploded.curr,
  exploded.id,
  exploded.name,
  exploded.price,
  exploded.promotion_info,
  exploded.qty,
  exploded.unit
FROM (
  SELECT
    customer_id,
    order_number,
    explode(parse_product_list(ordered_products)) AS exploded
  FROM databricks_simulated_retail_customer_data.v01.sales_orders
) t
limit 1

In [0]:
%sql
CREATE TABLE IF NOT EXISTS end_to_end_retail.db_landing.ordered_product (
  customer_id BIGINT,
  order_number STRING,
  curr STRING,
  id STRING,
  name STRING,
  price STRING,
  qty STRING,
  unity STRING
)
USING DELTA;

-- 2. Insert data from your source
INSERT INTO end_to_end_retail.db_landing.ordered_product (
  customer_id,
  order_number,
  curr,
  id,
  name,
  price,
  qty,
  unity
)
SELECT
  customer_id,
  order_number,
  exploded.curr as curr,
  exploded.id as id,
  exploded.name as name,
  exploded.price as price,
  exploded.qty as qty,
  exploded.unit as unity
FROM (
  SELECT
    customer_id,
    order_number,
    explode(parse_product_list(ordered_products)) AS exploded
  FROM databricks_simulated_retail_customer_data.v01.sales_orders
) t

### Volumes

- (dealing with streams)

In [0]:
# # Import necessary functions
# from pyspark.sql.functions import input_file_name, current_timestamp

# # Configuration
# source_path = "/mnt/raw-data/incoming/"
# checkpoint_path = "/mnt/checkpoints/autoloader_drift/"
# schema_location = "/mnt/schemas/autoloader_drift/"
# target_table = "bronze_sensor_data"

# # 1. Define the Auto Loader Stream
# df_stream = (spark.readStream
#     .format("cloudFiles")
#     .option("cloudFiles.format", "json")             # Source format (json, csv, parquet, etc.)
#     .option("cloudFiles.schemaLocation", schema_location)
#     .option("cloudFiles.inferColumnTypes", "true")   # Better type inference
#     .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # Automatic drift handling
#     .load(source_path)
#     .withColumn("source_file", input_file_name())    # Track source for audit
#     .withColumn("ingested_at", current_timestamp())
# )

# # 2. Write to Delta Table
# query = (df_stream.writeStream
#     .format("delta")
#     .outputMode("append")
#     .option("checkpointLocation", checkpoint_path)
#     .option("mergeSchema", "true")                  # Allow target table to evolve
#     .trigger(availableNow=True)                    # Process all new data and stop
#     .toTable(target_table)
# )

# query.awaitTermination()

#### retail-pipeline (customers)

2. Auto Loader

**Advantages:**

**Scalability:** Efficiently scales to discover and ingest millions or billions of files using cloud-native APIs, reducing discovery costs.

**Schema Handling:** Robust automatic schema inference and evolution support.

**Operational Ease:** Built on Spark Structured Streaming, it automatically tracks progress via checkpointing, removing the need for manual orchestration.

**Disadvantages:**

**Complexity:** Slightly more complex to set up and manage compared to COPY INTO (requires checkpoint locations, specific configurations).

**Reprocessing:** Harder to reprocess a select subset of files once ingested.

**Best Cases for Usage:** Continuous or near real-time incremental ingestion of files from cloud storage.
Handling data with frequently evolving schemas (JSON, semi-structured data). Any file-based ingestion where high volume and production reliability are key concerns.

**Overview ::**

(Python, Scala, SQL (cloudFiles format)):
Near real-time/Continuous (micro-batch or continuous mode)
Cloud object storage files	
Efficient, scalable file notification or directory listing
Automatic inference and evolution handling	
Manual compute management (runs on a cluster)




##### readStream

In [0]:
# Import necessary functions
from pyspark.sql.functions import input_file_name, current_timestamp

# Configuration
source_path = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/customer_auto/"  #only one for now
checkpoint_path = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/customer_auto/_checkpoint_stream"
schema_location = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/customer_auto/"
target_table = "end_to_end_retail.db_landing.customer_autoloader"

#### Simple example for one file only
- add other later

In [0]:
customers_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaHints", "timestamp DATE")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # Automatic drift handling
    .load(source_path)
    
    # .option("cloudFiles.useNotifications", "true")
    # .useNotifications will read from queues message coming from cloud services
    # AWS Event Notification, Azure Event Grid
    # .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
)

In [0]:
# 2. Write to Delta Table
query = (customers_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")                  # Allow target table to evolve
    .trigger(availableNow=True)                    # Process all new data and stop
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
query.stop()

##### readStream tracking columns

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.customer_autoloader limit 5

#### retail-pipeline (orders)

In [0]:
# Load JSON file
df = spark.read.json("/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/orders/stream_json/00.json")
display(df.head(5))

#### retail-pipeline (status)

In [0]:
# Load JSON file
df = spark.read.json("/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/status/stream_json/00.json")
display(df.head(5))

#### source_files (customers.csv)

In [0]:
# Load JSON file
df = spark.read.csv("/Volumes/databricks_simulated_retail_customer_data/v01/source_files/customers.csv", header=True)
display(df.head(5))

#### source_files (sales.csv)

In [0]:
# Load JSON file
df = spark.read.csv("/Volumes/databricks_simulated_retail_customer_data/v01/source_files/sales.csv", header=True)
display(df.head(5))

#### source_files (sales_orders.csv)

In [0]:
# Load JSON file
df = spark.read.csv("/Volumes/databricks_simulated_retail_customer_data/v01/source_files/sales_orders.csv", header=True)
display(df.head(5))

## V02
 


3. Spark Structured Streaming (Direct API)

Advantages:
Flexibility: Provides full control over cluster size, performance tuning, and custom logic (e.g., using foreachBatch for custom sinks).
Source Diversity: Can ingest from a wide range of sources beyond cloud files (Kafka, Kinesis, etc.).
Disadvantages:
Management Overhead: Requires manual management of infrastructure, checkpointing, and error handling.
Complexity: Requires more boilerplate code compared to DLT's declarative approach.
Best Cases for Usage:
SLA-sensitive streaming workloads that require granular control and specific performance tuning.
Ingesting data from sources not directly supported by DLT's declarative syntax (though DLT can often use readStream internally).
Scenarios needing highly custom data sinks or complex, non-standard processing logic.

#### business_daily_events

In [0]:
# Load JSON file
df = spark.read.json("/Volumes/databricks_simulated_retail_customer_data/v02/business_daily_events/retail_business_events_2025-11-01.json")
display(df.head(5))

#### customer_changes_daily

In [0]:
# Load JSON file
df = spark.read.json("/Volumes/databricks_simulated_retail_customer_data/v02/customer_changes_daily/customer_changes_2025-11-01.json")
display(df.head(5))

#### subsidiary_daily_orders

##### bright_home_orders

In [0]:
# Load CSV file
df = spark.read.csv("/Volumes/databricks_simulated_retail_customer_data/v02/subsidiary_daily_orders/bright_home_orders/bsh_orders_2025-11-01.csv", header=True)
display(df.limit(5))

##### lumina_sports_orders

In [0]:
# Load CSV file
df = spark.read.csv("/Volumes/databricks_simulated_retail_customer_data/v02/subsidiary_daily_orders/lumina_sports_orders/lms_orders_2025-11-01.csv", header=True)
display(df.limit(5))


##### northstar_outfitters_orders

In [0]:
# Load json file
df = spark.read.json("/Volumes/databricks_simulated_retail_customer_data/v02/subsidiary_daily_orders/northstar_outfitters_orders/nso_orders_2025-11-01.json")
display(df.limit(5))
